In [3]:
pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install transformers tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm


C:\Users\sara9\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [15]:
# ==========================================
# --- 1. 系統參數與辭典設定 ---
# ==========================================

INPUT_CSV = "NBA_News_2023_Full_Cohort.csv"
OUTPUT_CSV = "NBA_News_2023_Sentiment_Features.csv"


BASKETBALL_KEYWORDS = [
    'nba', 'basketball', 'draft', 'rookie', 'trade', 'contract', 'hoops', 
    'points', 'rebounds', 'assists', 'coach', 'ncaa', 'college', 'guard', 
    'forward', 'center', 'scout', 'prospect', 'g-league', 'g league', 
    'summer league', 'playoffs', 'championship', 'pick', 'roster', 'minutes',
    'hawks', 'celtics', 'nets', 'hornets', 'bulls', 'cavaliers', 'cavs', 
    'mavericks', 'mavs', 'nuggets', 'pistons', 'warriors', 'rockets', 
    'pacers', 'clippers', 'lakers', 'grizzlies', 'heat', 'bucks', 
    'timberwolves', 'wolves', 'pelicans', 'knicks', 'thunder', 'magic', 
    '76ers', 'sixers', 'suns', 'trail blazers', 'blazers', 'kings', 
    'spurs', 'raptors', 'jazz', 'wizards','injury'
]

# 🏥 分級傷病辭典 (醫療風險降維)
INJURY_DICT = {
    'severe': ['torn acl', 'achilles', 'microfracture', 'ruptured', 'season-ending', 
               'ligament tear', 'out for the season', 'torn achilles'],
    'moderate': ['meniscus', 'surgery', 'concussion', 'fracture', 'broken', 
                 'dislocated', 'hamstring tear', 'mcl', 'lcl'],
    'minor': ['sprain', 'bruise', 'soreness', 'tightness', 'day-to-day', 
              'tweak', 'spasms', 'contusion', 'cramp']
}

# ==========================================
# --- 2. 輔助判定函式 ---
# ==========================================
def is_basketball_related(text):
    text_lower = str(text).lower()
    return any(keyword in text_lower for keyword in BASKETBALL_KEYWORDS)

def is_player_in_title(player_name, content):
    # 將爬蟲組合的 "標題 | 摘要" 拆開，只取標題部分
    parts = str(content).split(' | ', 1)
    title = parts[0].lower()
    
    # 將球員名字拆開比對 (忽略 Jr. 或 II 等短字)
    name_parts = [part.lower() for part in str(player_name).split() if len(part) > 2]
    for part in name_parts:
        if part in title: return True
    return False

def extract_injury_severity(text):
    text_lower = str(text).lower()
    if any(keyword in text_lower for keyword in INJURY_DICT['severe']): return 3.0
    if any(keyword in text_lower for keyword in INJURY_DICT['moderate']): return 2.0
    if any(keyword in text_lower for keyword in INJURY_DICT['minor']): return 1.0
    return 0.0

# ==========================================
# --- 3. 模型載入與資料讀取 ---
# ==========================================
print("🧠 正在載入 RoBERTa 深度學習情緒分析模型...")
# 自動偵測是否可用 GPU 加速
device = 0 if torch.cuda.is_available() else -1
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device
)

try:
    df = pd.read_csv(INPUT_CSV)
    print(f"📂 成功讀取資料庫，共計 {len(df)} 筆新聞。")
except FileNotFoundError:
    print(f"❌ 找不到 {INPUT_CSV}，請確認檔名。")
    exit()

df['Content'] = df['Content'].fillna('').astype(str)

# ==========================================
# --- 4. 執行特徵萃取管線 ---
# ==========================================
print("\n🚀 開始執行【主角過濾 + 領域防護 + 情緒分析 + 傷病偵測】管線：")

sentiment_labels = []
sentiment_scores = []
injury_scores = []

filtered_out_basketball = 0 
filtered_out_title = 0      

for index, row in tqdm(df.iterrows(), total=len(df), desc="分析進度"):
    player_name = row['Player']
    text = row['Content']
    
    if not text.strip():
        sentiment_labels.append("irrelevant")
        sentiment_scores.append(0.0)
        injury_scores.append(0.0)
        continue
        
    # 🛡️ 濾網 1：主角聚焦 (踢掉只是順帶提起的陪襯新聞)
    if not is_player_in_title(player_name, text):
        sentiment_labels.append("irrelevant")
        sentiment_scores.append(0.0)
        injury_scores.append(0.0)
        filtered_out_title += 1
        continue
        
    # 🛡️ 濾網 2：實體消歧義 (踢掉同名同姓的社會/政治新聞)
    if not is_basketball_related(text):
        sentiment_labels.append("irrelevant") 
        sentiment_scores.append(0.0)
        injury_scores.append(0.0)
        filtered_out_basketball += 1
        continue
        
    # 🌟 擷取文字優化：盡量取 '|' 後面的內文摘要餵給模型
    parts = text.split(' | ', 1)
    if len(parts) > 1 and len(parts[1]) > 50:
        analysis_text = parts[1][:500] 
    else:
        analysis_text = text[:500]     

    try:
        # 🧠 算出連續極性分數 (突破中立海嘯)
        # 注意：強制模型吐出所有三個類別的機率
        results = sentiment_analyzer(analysis_text, top_k=None)
        if isinstance(results[0], list): results = results[0]
        
        pos_score = neg_score = neu_score = 0.0
        
        for item in results:
            if item['label'] == 'positive': pos_score = item['score']
            elif item['label'] == 'negative': neg_score = item['score']
            elif item['label'] == 'neutral': neu_score = item['score']
            
        polarity = pos_score - neg_score
        dominant_label = max(results, key=lambda x: x['score'])['label']
        
        # 🏥 傷病嚴重度掃描
        injury_severity = extract_injury_severity(text)
        
        sentiment_labels.append(dominant_label)
        sentiment_scores.append(round(polarity, 4))
        injury_scores.append(injury_severity)
        
    except Exception as e:
        print(f"\n⚠️ 解析錯誤: {e}")
        sentiment_labels.append("error")
        sentiment_scores.append(0.0)
        injury_scores.append(0.0)

# ==========================================
# --- 5. 特徵寫入與純淨化存檔 ---
# ==========================================
df['Sentiment_Label'] = sentiment_labels
df['Sentiment_Score'] = sentiment_scores
df['Injury_Severity_Score'] = injury_scores

# 🌟 剔除所有假新聞與無關新聞，只保留高純度特徵
df_clean = df[df['Sentiment_Label'] != 'irrelevant'].copy()
df_clean.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"🎉 大師級深度學習特徵萃取完成！")
print(f"🛡️ 標題聚焦：剔除了 {filtered_out_title} 篇非主角陪襯新聞。")
print(f"🛡️ 實體消歧義：擋下了 {filtered_out_basketball} 篇非籃球雜訊報導。")
print(f"📁 帶有【量化情緒分數】與【醫療風險分數】的極致純淨資料 ({len(df_clean)} 筆) 已儲存至: {OUTPUT_CSV}")
print("="*50)

🧠 正在載入 RoBERTa 深度學習情緒分析模型...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📂 成功讀取資料庫，共計 1275 筆新聞。

🚀 開始執行【主角過濾 + 領域防護 + 情緒分析 + 傷病偵測】管線：


分析進度: 100%|██████████| 1275/1275 [01:04<00:00, 19.67it/s]


🎉 大師級深度學習特徵萃取完成！
🛡️ 標題聚焦：剔除了 218 篇非主角陪襯新聞。
🛡️ 實體消歧義：擋下了 261 篇非籃球雜訊報導。
📁 帶有【量化情緒分數】與【醫療風險分數】的極致純淨資料 (796 筆) 已儲存至: NBA_News_2023_Sentiment_Features.csv


In [16]:
import pandas as pd

# ==========================================
# --- 1. 設定檔案名稱 ---
# ==========================================
# ⚠️ 請替換成你剛剛跑完情緒分析的 CSV 檔名
INPUT_CSV = "NBA_News_2023_Sentiment_Features.csv" 
OUTPUT_CSV = "NBA_News_2023_Player_Scores.csv"

# ==========================================
# --- 2. 讀取資料 ---
# ==========================================
try:
    df = pd.read_csv(INPUT_CSV)
    print(f"📂 成功讀取資料：共 {len(df)} 篇新聞紀錄。")
except FileNotFoundError:
    print(f"❌ 找不到 {INPUT_CSV}，請確認檔案名稱與路徑。")
    exit()

# 確保分數欄位是數值格式 (避免有字串混入導致無法加總)
df['Sentiment_Score'] = pd.to_numeric(df['Sentiment_Score'], errors='coerce')

print("⚙️ 正在將所有新聞按球員進行分組與計算...")

# ==========================================
# --- 3. 執行特徵聚合 (Groupby) ---
# ==========================================
# 將資料依照 'Player' 進行群組化，並計算不同的統計量
player_scores = df.groupby('Player').agg(
    Article_Count=('Sentiment_Score', 'count'),          # 總篇數 (媒體聲量)
    Total_Sentiment_Score=('Sentiment_Score', 'sum'),    # 總分 (所有新聞分數加總)
    Average_Sentiment_Score=('Sentiment_Score', 'mean')  # 平均分 (整體情緒期望值)
).reset_index()

# 將數字四捨五入到小數點後 4 位，讓資料表看起來更乾淨
player_scores = player_scores.round(4)

# 針對「平均分」由高到低進行排序
player_scores = player_scores.sort_values(by='Average_Sentiment_Score', ascending=False)

# ==========================================
# --- 4. 存檔與輸出 ---
# ==========================================
player_scores.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"🎉 計算完成！共處理了 {len(player_scores)} 位球員的最終分數。")
print(f"📁 每個人的專屬分數表已儲存至: {OUTPUT_CSV}")
print("="*50)

# 印出前五名與最後五名讓你可以直接檢查結果
print("\n🏆 【平均情緒最高】的前 5 名球員：")
print(player_scores.head(5).to_string(index=False))

print("\n📉 【平均情緒最低】的後 5 名球員：")
print(player_scores.tail(5).to_string(index=False))

📂 成功讀取資料：共 796 篇新聞紀錄。
⚙️ 正在將所有新聞按球員進行分組與計算...

🎉 計算完成！共處理了 19 位球員的最終分數。
📁 每個人的專屬分數表已儲存至: NBA_News_2023_Player_Scores.csv

🏆 【平均情緒最高】的前 5 名球員：
          Player  Article_Count  Total_Sentiment_Score  Average_Sentiment_Score
     Jalen Duren             61                14.0110                   0.2297
  Paolo Banchero             63                14.3833                   0.2283
   Mark Williams             22                 4.9116                   0.2233
MarJon Beauchamp             32                 6.5984                   0.2062
Jabari Smith Jr.             73                14.1612                   0.1940

📉 【平均情緒最低】的後 5 名球員：
             Player  Article_Count  Total_Sentiment_Score  Average_Sentiment_Score
      Dyson Daniels             33                 3.1994                   0.0970
TyTy Washington Jr.             12                 1.0158                   0.0846
     Walker Kessler             51                 3.4612                   0.0679
       Jake LaRavia      

In [17]:
import pandas as pd
import numpy as np
import os

INPUT_CSV = "NBA_News_2023_Sentiment_Features.csv" # 你跑完情緒與傷病分析的檔案
OUTPUT_CSV = "NBA_Player_Final_Master_Features.csv"  # 你已經建好球員名單的總表

print(f"📂 正在讀取情緒分析結果：{INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
df['Sentiment_Score'] = pd.to_numeric(df['Sentiment_Score'], errors='coerce')

# --- 1. 執行基礎聚合與傷病特徵 ---
def calculate_red_flag_ratio(scores):
    negative_count = sum(1 for score in scores if score < -0.3)
    return negative_count / len(scores) if len(scores) > 0 else 0

print("⚙️ 正在進行第一階段：特徵聚合...")
aggregated_df = df.groupby(['Draft_Year', 'Player']).agg(
    Article_Count=('Sentiment_Score', 'count'),                 
    Raw_Mean_Sentiment=('Sentiment_Score', 'mean'),                   
    Red_Flag_Ratio=('Sentiment_Score', calculate_red_flag_ratio), 
    Injury_Proneness_Index=('Injury_Severity_Score', 'sum') 
).reset_index()

# --- 2. 同梯次聲量標準化 (PR 值) ---
print("⚖️ 正在進行第二階段：計算同梯次聲量 PR 值...")
aggregated_df['Cohort_Buzz_PR'] = aggregated_df.groupby('Draft_Year')['Article_Count'].rank(pct=True)

# --- 3. 貝氏平滑化 (防守：校正小樣本偏差) ---
print("🛡️ 正在進行第三階段：貝氏平滑校正...")
global_mean = aggregated_df['Raw_Mean_Sentiment'].mean()
C_weight = aggregated_df['Article_Count'].median() 

aggregated_df['Adjusted_Sentiment'] = (
    (aggregated_df['Article_Count'] * aggregated_df['Raw_Mean_Sentiment']) + (C_weight * global_mean)
) / (aggregated_df['Article_Count'] + C_weight)

# --- 4. 🌟 球星價值指數 (進攻：絕對獎勵聲量優勢) ---
print("🚀 正在進行第四階段：計算球星價值指數 (Star Power Index)...")
aggregated_df['Star_Power_Index'] = aggregated_df['Cohort_Buzz_PR'] * aggregated_df['Adjusted_Sentiment']

# 四捨五入
aggregated_df = aggregated_df.round(4)

# ==========================================
# 🌟 5. 精準對齊填入現有總表 (Update 模式)
# ==========================================
print(f"🔄 正在將資料精準填入現有主表：{OUTPUT_CSV}...")
if os.path.exists(OUTPUT_CSV):
    df_master = pd.read_csv(OUTPUT_CSV)
    
    # 清理名稱前後的空白，確保能 100% 對齊
    df_master['Player'] = df_master['Player'].astype(str).str.strip()
    aggregated_df['Player'] = aggregated_df['Player'].astype(str).str.strip()

    # 將 Player 設為 Index，這樣 Pandas 就會自動用名字當作鑰匙對齊
    df_master.set_index('Player', inplace=True)
    aggregated_df.set_index('Player', inplace=True)

    # 定義我們要填進去的新欄位名稱
    feature_cols = [
        'Article_Count', 'Raw_Mean_Sentiment', 'Red_Flag_Ratio',
        'Injury_Proneness_Index',
        'Cohort_Buzz_PR', 'Adjusted_Sentiment', 'Star_Power_Index'
    ]

    # 防呆機制：如果你的主表裡面還沒有這些欄位的標題，先幫你建出空欄位
    for col in feature_cols:
        if col not in df_master.columns:
            df_master[col] = np.nan

    # 🌟 核心魔法：使用 .update()，它只會「更新/填入」對應名字與對應欄位的格子
    df_master.update(aggregated_df[feature_cols])

    # 填完之後把 Index 變回原本的 Player 欄位
    df_master.reset_index(inplace=True)

    # 存檔覆寫
    df_master.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"   - ✅ 填入完成！總表維持 {len(df_master)} 列不變。")

else:
    print(f"   ⚠️ 找不到 {OUTPUT_CSV}，請確認檔案名稱與路徑！")

print("\n" + "="*50)
print("🎉 任務完成！資料已完美填入你的球員名單格子中！")
print("="*50)

📂 正在讀取情緒分析結果：NBA_News_2023_Sentiment_Features.csv
⚙️ 正在進行第一階段：特徵聚合...
⚖️ 正在進行第二階段：計算同梯次聲量 PR 值...
🛡️ 正在進行第三階段：貝氏平滑校正...
🚀 正在進行第四階段：計算球星價值指數 (Star Power Index)...
🔄 正在將資料精準填入現有主表：NBA_Player_Final_Master_Features.csv...
   - ✅ 填入完成！總表維持 4669 列不變。

🎉 任務完成！資料已完美填入你的球員名單格子中！
